# **Exercises from the Portfolio PDF**

## Exercise 15

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import plot_tree, export_text, _classes
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import itertools

In [ ]:
dataset_path = 'https://raw.githubusercontent.com/Koldim2001/test_api/refs/heads/main/titanic.csv' # importing the dataset via a web link rather than file directory.
df = pd.read_csv(dataset_path) # defining the pandas dataframe as the url "dataset_path"
df.info() # print information about the dataframe to identify non-zero fields.

In [4]:
df = df[['Survived', 'Pclass', 'Age', 'Fare', 'SibSp', 'Sex', 'Parch', 'Embarked']] # defining the features of this model, we will remove the 'survived' column as this is the label/target.
df.info() # the new dataset after being changed. We can see that "Age" and "Embarked" features have null values so we will remove this in the next step.

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Age       714 non-null    float64
 3   Fare      891 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Sex       891 non-null    str    
 6   Parch     891 non-null    int64  
 7   Embarked  889 non-null    str    
dtypes: float64(2), int64(4), str(2)
memory usage: 55.8 KB


In [ ]:
df = df.dropna(subset=['Age','Embarked']) # dropping the "not availiable" values to clean the dataframe
df.info() # we can observe that the null values have been removed and now the dataset is "clean"

<class 'pandas.DataFrame'>
Index: 712 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  712 non-null    int64  
 1   Pclass    712 non-null    int64  
 2   Age       712 non-null    float64
 3   Fare      712 non-null    float64
 4   SibSp     712 non-null    int64  
 5   Sex       712 non-null    str    
 6   Parch     712 non-null    int64  
 7   Embarked  712 non-null    str    
dtypes: float64(2), int64(4), str(2)
memory usage: 50.1 KB


In [ ]:
df.isna().sum() # a list of the features that have null values, we can see that none of them do.

Survived    0
Pclass      0
Age         0
Fare        0
SibSp       0
Sex         0
Parch       0
Embarked    0
dtype: int64

In [8]:
df.drop(columns='Survived') # dropping the survived column as this will be our target/label.

,Pclass,Age,Fare,SibSp,Sex,Parch,Embarked
0,3,22.0,7.2500,1,male,0,S
1,1,38.0,71.2833,1,female,0,C
2,3,26.0,7.9250,0,female,0,S
3,1,35.0,53.1000,1,female,0,S
4,3,35.0,8.0500,0,male,0,S
...,...,...,...,...,...,...,...
885,3,39.0,29.1250,0,female,5,Q
886,2,27.0,13.0000,0,male,0,S
887,1,19.0,30.0000,0,female,0,S
889,1,26.0,30.0000,0,male,0,C


In [9]:
train, test = train_test_split(df, test_size=0.2) # splitting the dataset and using 20% (0.2) to train it.

In [10]:
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    Plots confusion matrix
    cm - confusion matrix
    classes - class list
    normalize - normalize to 1 if True
    title - plot title
    cmap - color map
    """

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

In [11]:
def experiment(max_depth, min_samples_split):
    """
    Builds and trains Decision Tree model
    """
    # Build and train Decision Tree model
    model = DecisionTreeClassifier(max_depth=max_depth, min_samples_split=min_samples_split, random_state=42)
    model.fit(train.drop('Survived', axis=1), train['Survived'])

    # Calculate accuracy metrics
    preds = model.predict(test.drop('Survived', axis=1))
    acc = accuracy_score(test['Survived'], preds)
    cm = confusion_matrix(test['Survived'], preds)

    print("accuracy", acc)

    # Plot confusion matrix
    plot_confusion_matrix(cm, classes=['Not Survived', 'Survived'])

    # Classification report
    report = classification_report(test['Survived'], preds, target_names=['Not Survived', 'Survived'])
    print(report)

    # Save model in pickle format
    with open('../models/model_dt.pkl', 'wb') as f:
        pickle.dump(model, f)
        